# Base Path

In [0]:
base_path = "/Volumes/weather_catalog/weather_platform/weather_data"

In [0]:
bronze_path = f"{base_path}/bronze"
silver_path = f"{base_path}/silver"

print(bronze_path)
print(silver_path)

# Imports

In [0]:
import json
import pandas as pd
from datetime import datetime
from pyspark.sql.functions import to_date, col, sum
from pyspark.sql.functions import year as spark_year

# Cities

In [0]:
cities = {
    "chennai": (13.0827, 80.2707),
    "coimbatore": (11.0168, 76.9558),
    "mumbai": (19.0760, 72.8777),
    "delhi": (28.6139, 77.2090),
    "bangalore": (12.9716, 77.5946),
    "hyderabad": (17.3850, 78.4867),
    "kolkata": (22.5726, 88.3639),
    "pune": (18.5204, 73.8567),
    "ahmedabad": (23.0225, 72.5714),
    "kochi": (9.9312, 76.2673)
}

# Open Meteo JSON into Tabular Records

## Inspection of data

In [0]:
sample_file = (
    f"{bronze_path}/open_meteo/"
    "city=chennai/year=2020/raw.json"
)

raw_json = dbutils.fs.head(sample_file)

data = json.loads(raw_json)

print(data["daily"].keys())

## Convert to dataframe

In [0]:
sample_file = (
    f"{bronze_path}/open_meteo/"
    "city=chennai/year=2020/raw.json"
)

raw_json = dbutils.fs.head(sample_file)

data = json.loads(raw_json)

daily = data["daily"]

pdf = pd.DataFrame({
    "date": daily["time"],
    "temperature_c": daily["temperature_2m_mean"],
    "precipitation_mm": daily["precipitation_sum"]
})

pdf["city"] = "chennai"
pdf["source"] = "open_meteo"

pdf.head()

## Conversion into spark df

In [0]:
spark_df = spark.createDataFrame(pdf)

display(spark_df)

## Open Meteo Silver df with all years/cities

In [0]:
# Empty list
all_records = []

# Read All Cities and Years
for city, coords in cities.items():

    for year in range(2020, 2025):

        file_path = (
            f"{bronze_path}/open_meteo/"
            f"city={city}/year={year}/raw.json"
        )

        raw_json = dbutils.fs.head(file_path)

        data = json.loads(raw_json)

        daily = data["daily"]

        for i in range(len(daily["time"])):

            all_records.append(
                (
                    daily["time"][i],
                    city,
                    "open_meteo",
                    float(daily["temperature_2m_mean"][i]),
                    float(daily["precipitation_sum"][i])
                )
            )

# Spark Dataframe
columns = [
    "date",
    "city",
    "source",
    "temperature_c",
    "precipitation_mm"
]

open_meteo_df = spark.createDataFrame(
    all_records,
    columns
)

display(open_meteo_df)

# Validation of row count
print(open_meteo_df.count())

## Data type check

In [0]:
open_meteo_df.printSchema()

## Datatype conversion of date

In [0]:
open_meteo_df = open_meteo_df.withColumn(
    "date",
    to_date("date", "yyyy-MM-dd")
)

open_meteo_df.printSchema()

# NASA Power JSON into Tabular Records

## Inspection of data

In [0]:
sample_file = (
    f"{bronze_path}/nasa_power/"
    "city=chennai/year=2020/raw.json"
)

raw_json = dbutils.fs.head(sample_file)

data = json.loads(raw_json)

print(data["properties"]["parameter"].keys())

## NASA Power SIlver df with all years/cities

In [0]:
nasa_records = []

# Read All Cities and Years
for city in cities.keys():

    for year in range(2020, 2025):

        file_path = (
            f"{bronze_path}/nasa_power/"
            f"city={city}/year={year}/raw.json"
        )

        raw_json = dbutils.fs.head(file_path)

        data = json.loads(raw_json)

        temperature = data["properties"]["parameter"]["T2M"]
        rainfall = data["properties"]["parameter"]["PRECTOTCORR"]

        # Conversion to date type
        for date_key in temperature.keys():

            nasa_records.append(
                (
                    datetime.strptime(date_key, "%Y%m%d").date(),
                    city,
                    "nasa_power",
                    float(temperature[date_key]),
                    float(rainfall[date_key])
                )
            )
# Spark Dataframe
columns = [
    "date",
    "city",
    "source",
    "temperature_c",
    "precipitation_mm"
]

nasa_power_df = spark.createDataFrame(
    nasa_records,
    columns
)

display(nasa_power_df)

# Validation of row count
print(nasa_power_df.count())

## Schema check

In [0]:
nasa_power_df.printSchema()

# Combine both the data sources

## Union dfs of 2 sources

In [0]:
combined_df = open_meteo_df.unionByName(nasa_power_df)

print(f"Total rows: {combined_df.count()}")

## Verify source's distribution

In [0]:
combined_df.groupBy("source").count().show()

# Missing Value check

In [0]:
combined_df.select(
    sum(col("date").isNull().cast("int")).alias("null_dates"),
    sum(col("city").isNull().cast("int")).alias("null_city"),
    sum(col("temperature_c").isNull().cast("int")).alias("null_temp"),
    sum(col("precipitation_mm").isNull().cast("int")).alias("null_rain")
).show()

# Duplication check

In [0]:
total_rows = combined_df.count()

distinct_rows = combined_df.distinct().count()

print(f"Total Rows    : {total_rows}")
print(f"Distinct Rows : {distinct_rows}")
print(f"Duplicates    : {total_rows - distinct_rows}")

# Final Silver Layer df

## Check combined df

In [0]:
silver_df = combined_df

print(f"Silver rows: {silver_df.count()}")

## Check year variable

In [0]:
print(year)

## Add year col (for partition)

In [0]:
silver_df = silver_df.withColumn(
    "year",
    spark_year("date")
)

silver_df.printSchema()
display(silver_df.limit(5))

## Write data as parquet

In [0]:
silver_df.write \
    .mode("overwrite") \
    .partitionBy("year") \
    .parquet(silver_path)

print("Silver dataset saved successfully")

# Check Silver Layer

In [0]:
display(dbutils.fs.ls(silver_path))

In [0]:
spark.read.parquet(silver_path).count()